<a href="https://colab.research.google.com/github/luizalmin-netizen/js-developer-pokedex/blob/main/Explorando_IA_Generativa_em_um_Pipeline_de_ETL_com_Python.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Extração (E) - Lendo dados de um arquivo CSV

Nesta etapa, simularemos a extração de dados de clientes a partir de um arquivo CSV. Como não temos um arquivo CSV fornecido, vamos criar um arquivo de exemplo com dados fictícios para demonstrar o processo.

In [1]:
import pandas as pd
import io

# Criar dados fictícios para o CSV
data = {
    'Nome': ['João', 'Maria', 'Pedro', 'Ana', 'Carlos'],
    'Conta': ['12345-6', '67890-1', '11223-3', '44556-6', '77889-9'],
    'Cartão': ['**** **** **** 1234', '**** **** **** 5678', '**** **** **** 9012', '**** **** **** 3456', '**** **** **** 7890']
}

df_raw = pd.DataFrame(data)

# Salvar o DataFrame em um arquivo CSV temporário
csv_file_path = 'clientes.csv'
df_raw.to_csv(csv_file_path, index=False)

print(f"Arquivo '{csv_file_path}' criado com sucesso!")

Arquivo 'clientes.csv' criado com sucesso!


Agora, vamos ler o arquivo CSV que acabamos de criar para um DataFrame do pandas.

In [2]:
# Ler o arquivo CSV para um DataFrame
df_clientes = pd.read_csv(csv_file_path)

# Exibir as primeiras linhas do DataFrame
print("Dados dos clientes extraídos:")
display(df_clientes.head())

Dados dos clientes extraídos:


,Nome,Conta,Cartão
0,João,12345-6,**** **** **** 1234
1,Maria,67890-1,**** **** **** 5678
2,Pedro,11223-3,**** **** **** 9012
3,Ana,44556-6,**** **** **** 3456
4,Carlos,77889-9,**** **** **** 7890


## Transformação (T) - Gerando mensagens personalizadas com IA

Nesta etapa, usaremos um modelo de linguagem (como o GPT da OpenAI) para gerar mensagens personalizadas para cada cliente. Isso simula o processo de agregar valor aos dados brutos.

In [3]:
import os
from openai import OpenAI

# Configurar sua chave de API do OpenAI
# É uma boa prática armazenar chaves de API em variáveis de ambiente ou usar ferramentas de gerenciamento de segredos.
# Por simplicidade neste exemplo, você pode inseri-la diretamente, mas tenha cuidado ao compartilhar seu notebook.

os.environ['OPENAI_API_KEY'] = 'SUA_API_KEY_AQUI' # <-- Substitua pela sua chave de API real
client = OpenAI()

### Configurando a Chave da API OpenAI com o Secrets Manager (Recomendado)

Para maior segurança, é altamente recomendável usar o Gerenciador de Segredos do Colab para armazenar suas chaves de API. Siga estes passos:

1.  **Abra o painel de Segredos:** Clique no ícone de uma chave (🔑) na barra lateral esquerda do Colab.
2.  **Adicione um novo segredo:** Clique em "Adicionar um novo segredo".
3.  **Nomeie o segredo:** Defina o nome do segredo como `OPENAI_API_KEY` (é crucial que o nome seja exatamente este para que o código abaixo funcione).
4.  **Cole sua chave:** No campo "Valor", cole sua chave de API real da OpenAI.
5.  **Salve:** Certifique-se de que a opção "Acesso ao notebook" esteja ativada para este segredo.

Após configurar o segredo, execute a célula abaixo para carregar a chave de API no ambiente do notebook.

In [7]:
# Importa a biblioteca para gerenciar segredos no Colab
from google.colab import userdata
import os

# Obtenha a chave da API OpenAI do Secrets Manager do Colab
OPENAI_API_KEY = userdata.get('OPENAI_API_KEY')

# Verifique se a chave foi carregada e configure a variável de ambiente
if OPENAI_API_KEY:
    os.environ['OPENAI_API_KEY'] = OPENAI_API_KEY
    print("Chave da API OpenAI carregada com sucesso do Secrets Manager!")
else:
    print("ERRO: A chave 'OPENAI_API_KEY' não foi encontrada no Secrets Manager do Colab ou está vazia.")
    print("Por favor, configure-a conforme as instruções acima.")
    # Pode ser útil levantar um erro ou impedir a execução se a chave for crítica
    # raise ValueError("OPENAI_API_KEY não configurada no Secrets Manager.")


SecretNotFoundError: Secret OPENAI_API_KEY does not exist.

### Teste Rápido de Conexão com a OpenAI

Esta célula tentará listar os modelos disponíveis. Se for bem-sucedida, sua chave de API está funcionando.

In [6]:
try:
    # Tenta listar os modelos disponíveis
    models = client.models.list()
    print("Conexão com a OpenAI estabelecida com sucesso! Modelos disponíveis:")
    for model in models.data[:5]: # Exibe os primeiros 5 modelos para verificar
        print(f"- {model.id}")
except Exception as e:
    print(f"Falha na conexão com a OpenAI: {e}")
    print("Por favor, verifique sua chave de API na célula `74565309` e execute-a novamente.")

Falha na conexão com a OpenAI: Error code: 401 - {'error': {'message': 'Incorrect API key provided: SUA_API_****AQUI. You can find your API key at https://platform.openai.com/account/api-keys.', 'type': 'invalid_request_error', 'param': None, 'code': 'invalid_api_key'}}
Por favor, verifique sua chave de API na célula `74565309` e execute-a novamente.


Agora, vamos definir a função que fará a chamada à API da OpenAI para gerar a mensagem e, em seguida, aplicá-la a cada cliente no DataFrame.

In [4]:
def generate_ai_news(client, user):
    completion = client.chat.completions.create(
        model="gpt-3.5-turbo",
        messages=[
            {
                "role": "system",
                "content": "Você é um especialista em marketing bancário. Crie mensagens personalizadas para clientes com base em suas informações financeiras.",
            },
            {
                "role": "user",
                "content": f"Crie uma mensagem de marketing personalizada para o cliente {user['Nome']}, que possui uma conta {user['Conta']} e um cartão final {user['Cartão']}. A mensagem deve ser concisa e amigável. Destaque um benefício que o banco oferece.",
            },
        ],
    )
    return completion.choices[0].message.content.strip()

print("Gerando mensagens personalizadas para cada cliente...")
# Aplica a função de geração de notícias a cada linha do DataFrame
# Isso pode demorar um pouco, dependendo do número de clientes e da latência da API.
df_clientes['Mensagem'] = df_clientes.apply(lambda row: generate_ai_news(client, row), axis=1)

print("Mensagens geradas com sucesso!")
display(df_clientes.head())

Gerando mensagens personalizadas para cada cliente...


AuthenticationError: Error code: 401 - {'error': {'message': 'Incorrect API key provided: SUA_API_****AQUI. You can find your API key at https://platform.openai.com/account/api-keys.', 'type': 'invalid_request_error', 'param': None, 'code': 'invalid_api_key'}}

### Testando a integração com a OpenAI

Vamos testar a função `generate_ai_news` com um único cliente para garantir que a integração esteja funcionando corretamente.

In [5]:
# Selecionar o primeiro cliente para teste
first_client = df_clientes.iloc[0]

print(f"Testando com o cliente: {first_client['Nome']}")

# Gerar mensagem para o primeiro cliente
try:
    test_message = generate_ai_news(client, first_client)
    print("\nMensagem gerada com sucesso para o primeiro cliente:")
    print(test_message)
    print("\nSe a mensagem acima foi gerada corretamente, sua chave de API está funcionando!")
    print("Você pode então executar a célula anterior (e6ceb562) novamente para gerar as mensagens para todos os clientes.")
except Exception as e:
    print(f"\nOcorreu um erro ao gerar a mensagem para o cliente de teste: {e}")
    print("Por favor, verifique se sua chave de API está correta na célula `74565309` e se você a executou novamente.")

Testando com o cliente: João

Ocorreu um erro ao gerar a mensagem para o cliente de teste: Error code: 401 - {'error': {'message': 'Incorrect API key provided: SUA_API_****AQUI. You can find your API key at https://platform.openai.com/account/api-keys.', 'type': 'invalid_request_error', 'param': None, 'code': 'invalid_api_key'}}
Por favor, verifique se sua chave de API está correta na célula `74565309` e se você a executou novamente.
